# 🏆 Agent-Based Financial QA Challenge (UTN FRSR)
## Track Open-Weight en Google Colab (GPU T4 · Microservicio Caja Negra + Cloudflare Tunnel)

> **⚠️ Regla de oro del flujo de trabajo:** **En Colab NO se desarrolla.** Desarrollá localmente en tu PC/host sobre el repositorio propio de tu equipo, hacé `git commit` + `git push` a la rama principal en GitHub (repositorio público o privado con `GITHUB_TOKEN`), y usá este notebook únicamente como **runtime GPU T4** (`clone`/`pull`).

> **🔒 ¿Usás un repositorio privado?** (recomendado para no exponer tu solución a otros equipos):
> 1. En GitHub: *Settings → Developer settings → Personal access tokens → Fine-grained tokens* → creá un token de solo lectura (**`Contents: Read-only`**) para el repo de tu equipo.
> 2. En Colab: abrí **Secrets** (ícono de la llave 🔑 en la barra lateral izquierda), creá el secreto `GITHUB_TOKEN` con ese token y activá el acceso para este notebook.
> 3. En la **Celda 1**, marcá `USE_PRIVATE_REPO = True`.

Flujo en 5 celdas:
1. Clona (o actualiza con `git pull`) el **repositorio propio de tu equipo** (`REPO_URL`), define `sync_repo()`, instala dependencias y descarga `cloudflared`.
2. Instala `llama-cpp-python` con soporte CUDA (`cu122`) y registra las librerías compartidas de CUDA 12 requeridas por Colab T4.
3. Ejecuta `sync_repo()`, levanta (o reinicia recargando `src/`) el servidor FastAPI (`src/server.py`) en un hilo secundario dentro del mismo kernel (compartiendo VRAM con tu modelo GGUF) y verifica `GET /health`.
4. Ejecuta la evaluación local (`scripts/run_local_eval.py`) contra `data/dev_set.json` con parámetros configurables (`LIMIT` y `VERBOSE`) antes de consumir cuota remota.
5. Abre (o renueva sin reiniciar el servidor) el túnel público **Cloudflare Tunnel (`cloudflared`)** reutilizando `scripts/start_tunnel.py`.

In [ ]:
#@title 1. Clonar/actualizar el repositorio de tu equipo, instalar dependencias y descargar cloudflared { display-mode: "form" }
import base64
import os
import subprocess
from pathlib import Path

# ⚠️ Reemplazá por la URL HTTPS de GitHub del repositorio PROPIO de tu equipo.
# Si el repo es PRIVADO: creá un Fine-grained PAT (Contents: Read-only) en GitHub,
# guardalo en Colab Secrets como GITHUB_TOKEN (ícono 🔑 a la izquierda) y marcá USE_PRIVATE_REPO = True.
REPO_URL = "https://github.com/tu-usuario/el-repositorio.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
USE_PRIVATE_REPO = False #@param {type:"boolean"}
REPO_DIR = Path("/content/team-repo")

if "tu-usuario" in REPO_URL or "el-repositorio" in REPO_URL:
    raise ValueError("Configurá REPO_URL con la URL real del repositorio propio de tu equipo.")

_GIT_AUTH_ARGS = []
if USE_PRIVATE_REPO:
    if not REPO_URL.startswith("https://"):
        raise ValueError("Para USE_PRIVATE_REPO=True, REPO_URL debe comenzar con 'https://'.")
    from google.colab import userdata
    _token = (userdata.get("GITHUB_TOKEN") or "").strip()
    if not _token:
        raise ValueError(
            "Activaste USE_PRIVATE_REPO=True pero no se encontró el secreto 'GITHUB_TOKEN'. "
            "Cargalo en Colab Secrets (ícono 🔑 a la izquierda) y habilitá el acceso al notebook."
        )
    _basic = base64.b64encode(f"x-access-token:{_token}".encode()).decode()
    _GIT_AUTH_ARGS = ["-c", f"http.extraHeader=Authorization: Basic {_basic}"]

def _run_git(args, cwd=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    res = subprocess.run(["git", *_GIT_AUTH_ARGS, *args], cwd=cwd, env=env, capture_output=True, text=True)
    if res.returncode != 0:
        err = (res.stderr or res.stdout or "").strip()
        raise RuntimeError(
            f"Error al ejecutar git {args[0]}. Verificá REPO_URL, BRANCH y (si es privado) tu GITHUB_TOKEN.\n{err}"
        )

def sync_repo():
    """Sincroniza el repositorio del equipo con origin/{BRANCH}."""
    _run_git(["fetch", "origin", BRANCH], cwd=str(REPO_DIR))
    _run_git(["checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=str(REPO_DIR))
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=str(REPO_DIR), text=True).strip()
    print(f"🔄 Sincronizado con origin/{BRANCH} ({sha})")

if (REPO_DIR / ".git").exists():
    _run_git(["remote", "set-url", "origin", REPO_URL], cwd=str(REPO_DIR))
    os.chdir(REPO_DIR)
    sync_repo()
else:
    _run_git(["clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

!pip install -q -r requirements.txt

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ Descargando binario de Cloudflare Tunnel (cloudflared)...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

print("✅ Entorno base listo en", Path.cwd())


In [ ]:
#@title 2. Instalar llama-cpp-python (CUDA) y registrar librerías CUDA 12 en GPU T4 { display-mode: "form" }
import ctypes
import glob
import subprocess
from pathlib import Path

try:
    import llama_cpp  # noqa: F401
    print("✅ llama-cpp-python ya instalado.")
except ImportError:
    print("⏳ Instalando llama-cpp-python con aceleración CUDA para GPU T4...")
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

!pip install -q huggingface_hub

# Registrar libcudart.so.12 y libcublas.so.12 en ldconfig y precargar con RTLD_GLOBAL para que libllama.so las resuelva en Colab
cudart_hits = sorted(glob.glob("/usr/local/**/libcudart.so.12*", recursive=True))
if not cudart_hits:
    print("⏳ Instalando librerías runtime de CUDA 12 (nvidia-cuda-runtime-cu12, nvidia-cublas-cu12)...")
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    cudart_hits = sorted(glob.glob("/usr/local/**/libcudart.so.12*", recursive=True))

cublas_hits = sorted(glob.glob("/usr/local/**/libcublas*.so.12*", recursive=True))
lib_dirs = sorted({str(Path(p).parent) for p in cudart_hits} | {str(Path(p).parent) for p in cublas_hits})
if lib_dirs:
    Path("/etc/ld.so.conf.d/colab_cuda12.conf").write_text("\n".join(lib_dirs) + "\n")
    subprocess.run(["ldconfig"], check=False)
    for lib_dir in lib_dirs:
        for name in ("libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"):
            candidate = Path(lib_dir) / name
            if candidate.exists():
                ctypes.CDLL(str(candidate), mode=ctypes.RTLD_GLOBAL)

print("✅ CUDA 12 libs listas desde:", lib_dirs)
print("ℹ️ Esta celda solo prepara el runtime CUDA/llama-cpp-python; la descarga e instanciación del modelo ocurre dentro de tu código en src/ (podés detectar Colab leyendo os.getenv('RUNTIME_ENV') == 'colab').")
print("💡 Recomendación T4: cargar GGUF en archivo único (ej. bartowski/Qwen2.5-7B-Instruct-GGUF / Qwen2.5-7B-Instruct-Q4_K_M.gguf) con n_gpu_layers=-1, n_ctx=16384 y temperature=0.0.")


In [ ]:
#@title 3. Actualizar repo y levantar (o reiniciar) servidor FastAPI en hilo daemon { display-mode: "form" }
import gc
import os
import sys
import threading
import time
from pathlib import Path
import httpx
import uvicorn

if "sync_repo" not in globals():
    raise RuntimeError("Ejecutá primero la Celda 1 para clonar/sincronizar el repositorio de tu equipo.")
sync_repo()

# Señal estándar para que src/server.py y tus módulos detecten que corren en Google Colab
os.environ.setdefault("RUNTIME_ENV", "colab")

# Si ya había un servidor corriendo, lo detenemos y limpiamos módulos de src/ para recargar cambios del pull
if "uvicorn_server" in globals() and uvicorn_server is not None:
    print("🔄 Reiniciando servidor FastAPI...")
    uvicorn_server.should_exit = True
    if "server_thread" in globals() and server_thread is not None:
        server_thread.join(timeout=5)
    uvicorn_server = None
    server_thread = None
    app = None
    for mod in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
        del sys.modules[mod]
    gc.collect()

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from src.server import app

# Levantamos uvicorn en un thread daemon dentro del mismo proceso CUDA/T4 del kernel
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
uvicorn_server = uvicorn.Server(config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

# Esperar hasta 20s a que /health responda HTTP 200
health_payload = None
for _ in range(40):
    try:
        r = httpx.get("http://127.0.0.1:8000/health", timeout=2.0)
        if r.status_code == 200:
            health_payload = r.json()
            break
    except Exception:
        pass
    time.sleep(0.5)

if health_payload is None:
    raise RuntimeError("El servidor FastAPI no respondió en http://127.0.0.1:8000/health luego de 20s.")
print("✅ Healthcheck local:", health_payload)
print("💡 Re-ejecutá esta celda para bajar cambios nuevos y reiniciar el server (si aparece error de VRAM/CUDA, reiniciá la sesión de Colab).")


In [ ]:
#@title 4. Probar evaluación local contra data/dev_set.json { display-mode: "form" }
LIMIT = 5 #@param {type:"integer"}
VERBOSE = False #@param {type:"boolean"}

# LIMIT = 0 ejecuta los 150 casos completos de data/dev_set.json
flags = []
if LIMIT > 0:
    flags.extend(["--limit", str(LIMIT)])
if VERBOSE:
    flags.append("--verbose")
args_str = " ".join(flags)

print(f"▶ Ejecutando: python scripts/run_local_eval.py {args_str}".strip())
!python scripts/run_local_eval.py {args_str}


In [ ]:
#@title 5. Abrir (o renovar) Cloudflare Tunnel { display-mode: "form" }
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from scripts.start_tunnel import check_local_health, start_tunnel, stop_tunnel

ok, detail = check_local_health(host="127.0.0.1", port=8000)
if not ok:
    raise RuntimeError(
        f"El servidor FastAPI no respondió en http://127.0.0.1:8000/health ({detail}). "
        "Ejecutá primero la Celda 3 para levantar uvicorn."
    )

# Cerrar túnel previo si se re-ejecuta la celda
if "tunnel_proc" in globals() and tunnel_proc is not None and tunnel_proc.poll() is None:
    stop_tunnel(tunnel_proc)

# Abrir túnel público gratuito de Cloudflare (reutiliza scripts/start_tunnel.py)
tunnel_proc, public_url = start_tunnel(host="127.0.0.1", port=8000)

print("\n======================================================")
print("🌐 URL PÚBLICA PARA EL EVALUADOR REMOTO (Track Open-Weight):")
print(public_url)
print("======================================================\n")
